In [2]:

# --- Environment setup ---
# If running in a fresh environment, uncomment:
# !pip install datasets torch matplotlib

import re
import io
import json
import math
import random
from collections import Counter, defaultdict
from pathlib import Path

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)
print(f"Using device: {DEVICE}")

# Flip to False once you have internet access and want the real HF corpora.
OFFLINE_MODE = False

# Requested corpus size per class once running against the real HF datasets (OFFLINE_MODE=False).
# Set above 8,000 to leave headroom: loaders request this many *candidate* rows, but the final
# balanced count per class = min(deduped yield across all 4 sources), so a buffer above your
# actual target absorbs losses from de-duplication and length filtering.
TARGET_SAMPLES_PER_CLASS = 10000

LABELS = ["personal", "semantic", "working", "episodic"]
LABEL2IDX = {l: i for i, l in enumerate(LABELS)}
IDX2LABEL = {i: l for l, i in LABEL2IDX.items()}

OUT_DIR = Path("artifacts")
OUT_DIR.mkdir(exist_ok=True)


Using device: cpu



## Phase 1 — Data Aggregation

Each loader below pulls a real Hugging Face dataset, extracts the field that matches the
target linguistic profile, normalizes it, and returns a deduplicated list of clean strings.

**Design choice — stopwords are kept, not stripped.** Pronouns and function words
(`i`, `my`, `you`, `we`) are one of the strongest signals separating `personal`/`episodic`
(first-person) from `semantic` (third-person/objective) and `working` (imperative, no
grammatical subject). Removing them the way a generic text-classification pipeline would
throws away exactly the signal this classifier depends on.


In [3]:

# --- Text normalization utilities ---

MIN_CHARS, MAX_CHARS = 8, 300

def clean_text(t: str) -> str:
    if t is None:
        return ""
    t = t.replace("_comma_", ",")          # empathetic_dialogues artifact
    t = re.sub(r"\s+", " ", t).strip()
    return t

def is_valid(t: str) -> bool:
    return bool(t) and MIN_CHARS <= len(t) <= MAX_CHARS


In [4]:

# --- Real Hugging Face loaders (require internet access; used when OFFLINE_MODE = False) ---

def load_personal_memory(limit=10000):
    '''Personal Memory: first-person identity/trait statements.
    Source: bavard/personachat_truecased -> 'personality' field
    (e.g. "i am a vegetarian.", "i have two dogs.")'''
    from datasets import load_dataset
    ds = load_dataset("bavard/personachat_truecased", split="train")
    texts, seen = [], set()
    for row in ds:
        for sent in row["personality"]:
            t = clean_text(sent)
            if is_valid(t) and t not in seen:
                seen.add(t); texts.append(t)
        if len(texts) >= limit:
            break
    return texts[:limit]

def load_semantic_memory(limit=10000):
    '''Semantic Memory: objective, encyclopedic facts & definitions.
    Source: wiki_qa -> 'answer' field (candidate sentences drawn from Wikipedia).'''
    from datasets import load_dataset
    ds = load_dataset("wiki_qa", split="train")
    texts, seen = [], set()
    for row in ds:
        t = clean_text(row["answer"])
        if is_valid(t) and t not in seen:
            seen.add(t); texts.append(t)
        if len(texts) >= limit:
            break
    return texts[:limit]

def load_working_memory(limit=10000):
    '''Working Memory: immediate, task-oriented, goal-driven commands.
    Source: tatsu-lab/alpaca -> 'instruction' field (imperative directives).'''
    from datasets import load_dataset
    ds = load_dataset("tatsu-lab/alpaca", split="train")
    texts, seen = [], set()
    for row in ds:
        t = clean_text(row["instruction"])
        if is_valid(t) and t not in seen:
            seen.add(t); texts.append(t)
        if len(texts) >= limit:
            break
    return texts[:limit]

def load_episodic_memory(limit=10000):
    '''Episodic Memory: past-tense, situational conversational events.
    Source: empathetic_dialogues -> 'prompt' field (first-person narrated situations).'''
    from datasets import load_dataset
    ds = load_dataset("empathetic_dialogues", split="train")
    texts, seen = [], set()
    for row in ds:
        t = clean_text(row["prompt"])
        if is_valid(t) and t not in seen:
            seen.add(t); texts.append(t)
        if len(texts) >= limit:
            break
    return texts[:limit]

HF_LOADERS = {
    "personal": load_personal_memory,
    "semantic": load_semantic_memory,
    "working": load_working_memory,
    "episodic": load_episodic_memory,
}


In [5]:

# --- Offline synthetic fallback (only used when OFFLINE_MODE = True) ---
# Templated generators that mimic each source's linguistic profile, so the rest of the
# notebook (vectorizer, MLP, training, evaluation) can be exercised without internet access.

def _build_synthetic_corpus(seed=SEED, n_per_class=500):
    rng = random.Random(seed)

    personal_templates = [
        "I am a {adj} person who loves {hobby}.",
        "I have {n} {pet}s and I adore them.",
        "My favorite {thing} is {value}.",
        "I work as a {job} and I am proud of it.",
        "I grew up in {place} and it shaped who I am.",
        "I consider myself {adj} and {adj2}.",
    ]
    semantic_templates = [
        "The {concept} is defined as the {def_}.",
        "{Concept} refers to a process by which {process}.",
        "In physics, {concept} describes the relationship between {a} and {b}.",
        "The capital of {country} is {city}.",
        "{Concept} was first described in {year} by researchers studying {field}.",
        "A {concept} is a type of {category} found in {field}.",
    ]
    working_templates = [
        "Set a reminder for {time}.",
        "Book a table for {n} at {place}.",
        "Turn off the {device} in the living room.",
        "Add {item} to my shopping list.",
        "Schedule a meeting with {person} tomorrow at {time}.",
        "Find the nearest {place} and send me directions.",
    ]
    episodic_templates = [
        "Yesterday I went to the {place} and met {person}.",
        "Last week I was feeling {adj} after the {event}.",
        "We celebrated my {relation}'s birthday at {place} last month.",
        "I remember when I first {action} back in {year}.",
        "During the trip, I visited {place} and it was {adj}.",
        "This morning I {action} before heading to work.",
        "Last summer I {action} with {person} in {place}.",
        "A few months ago I {action} and it felt {adj}.",
        "Back in {year} I {action} while visiting {place}.",
        "Earlier this year I {action} with my {relation}.",
    ]
    fillers = {
        "adj": ["curious","happy","nervous","excited","calm","tired","anxious","confident"],
        "adj2": ["patient","stubborn","optimistic","organized"],
        "hobby": ["hiking","painting","chess","cooking","reading","gardening"],
        "n": ["two","three","four","five","six"],
        "pet": ["dog","cat","bird","fish","rabbit"],
        "thing": ["color","food","season","movie","book"],
        "value": ["blue","pizza","autumn","a sci-fi film","a mystery novel"],
        "job": ["teacher","engineer","nurse","artist","chef"],
        "place": ["Paris","the mountains","a small town","New York","the beach","the library","a cafe"],
        "concept": ["photosynthesis","gravity","inflation","democracy","osmosis","entropy"],
        "Concept": ["Photosynthesis","Gravity","Inflation","Democracy","Osmosis","Entropy"],
        "def_": ["conversion of light into chemical energy","force attracting masses",
                 "rise in prices over time","measure of disorder in a system"],
        "process": ["plants convert sunlight into energy","cells divide",
                    "water moves across a membrane","heat flows between two bodies"],
        "a": ["mass","voltage","supply","pressure"],
        "b": ["acceleration","current","demand","volume"],
        "category": ["organelle","economic indicator","political system","physical law"],
        "country": ["France","Japan","Germany","Brazil","Canada"],
        "city": ["Paris","Tokyo","Berlin","Brasilia","Ottawa"],
        "year": ["1928","1953","1687","2001","1867"],
        "field": ["genetics","chemistry","astronomy","economics"],
        "time": ["3pm","9am","noon","6:30pm","8am"],
        "device": ["lights","TV","speaker","heater","fan"],
        "item": ["milk","eggs","bread","coffee","rice"],
        "person": ["Sarah","the client","Dr. Lee","my manager","the team"],
        "event": ["exam","interview","storm","deadline"],
        "relation": ["sister","brother","mom","friend","cousin"],
        "action": ["rode a bike","cooked dinner","flew on a plane","went for a run","read a book"],
    }

    def fill(t):
        out = t
        for k, vals in fillers.items():
            if "{" + k + "}" in out:
                out = out.replace("{" + k + "}", rng.choice(vals))
        return out

    def gen(templates, n):
        return [fill(rng.choice(templates)) for _ in range(n)]

    return {
        "personal": gen(personal_templates, n_per_class),
        "semantic": gen(semantic_templates, n_per_class),
        "working": gen(working_templates, n_per_class),
        "episodic": gen(episodic_templates, n_per_class),
    }


In [6]:

# --- Build the unified corpus ---

if OFFLINE_MODE:
    print("OFFLINE_MODE=True -> using synthetic stand-in corpus (no internet required).")
    raw_data = _build_synthetic_corpus(n_per_class=600)
else:
    print("OFFLINE_MODE=False -> pulling real corpora from the Hugging Face Hub...")
    raw_data = {label: fn(limit=TARGET_SAMPLES_PER_CLASS) for label, fn in HF_LOADERS.items()}

for label, texts in raw_data.items():
    print(f"{label:10s}: {len(texts)} raw samples")

# Balance classes by undersampling to the smallest class, so the MLP isn't biased
# toward whichever HF source happened to be largest.
min_count = min(len(v) for v in raw_data.values())
print(f"\nBalancing all classes to {min_count} samples each")

rng = random.Random(SEED)
records = []
for label, texts in raw_data.items():
    sampled = rng.sample(texts, min_count)
    records.extend({"text": t, "label": label} for t in sampled)

rng.shuffle(records)
print(f"Total unified corpus: {len(records)} records")

with open(OUT_DIR / "corpus.json", "w") as f:
    json.dump(records, f, indent=2)


OFFLINE_MODE=False -> pulling real corpora from the Hugging Face Hub...


RuntimeError: Dataset scripts are no longer supported, but found personachat_truecased.py

In [ ]:

# --- Stratified train / val / test split (pure Python, no sklearn) ---

def stratified_split(records, val_frac=0.15, test_frac=0.15, seed=SEED):
    by_label = defaultdict(list)
    for r in records:
        by_label[r["label"]].append(r)

    train, val, test = [], [], []
    split_rng = random.Random(seed)
    for label, items in by_label.items():
        split_rng.shuffle(items)
        n = len(items)
        n_val = int(n * val_frac)
        n_test = int(n * test_frac)
        val.extend(items[:n_val])
        test.extend(items[n_val:n_val + n_test])
        train.extend(items[n_val + n_test:])

    split_rng.shuffle(train); split_rng.shuffle(val); split_rng.shuffle(test)
    return train, val, test

train_records, val_records, test_records = stratified_split(records)
print(f"train={len(train_records)}  val={len(val_records)}  test={len(test_records)}")


In [ ]:

# --- Sanity check: class balance + example rows ---

fig, ax = plt.subplots(figsize=(5, 3))
counts = [sum(1 for r in records if r["label"] == l) for l in LABELS]
ax.bar(LABELS, counts, color=["#4C72B0", "#DD8452", "#55A868", "#C44E52"])
ax.set_title("Class balance (full corpus)")
ax.set_ylabel("count")
plt.tight_layout()
plt.show()

print("\nSample records:")
for label in LABELS:
    example = next(r["text"] for r in records if r["label"] == label)
    print(f"  [{label:10s}] {example}")



## Phase 2 — Custom TF-IDF Vectorizer (pure Python, no sklearn)

Standard TF-IDF, built from scratch with `re` + `collections.Counter` + `math.log`:

- **Tokenizer**: lowercase word tokens + bigrams (`use_bigrams=True`) — bigrams capture
  short imperative phrases like `"turn_off"` or `"schedule_a"` that unigrams alone miss.
- **Vocabulary pruning**: `min_df` / `max_df_ratio` drop terms that are too rare (noise) or
  too common (uninformative across all four classes), then keep the top `max_features` by
  document frequency.
- **Weighting**: sublinear TF (`1 + log(tf)`) dampens the effect of a term repeated many
  times in one document, combined with smoothed IDF (`ln((1+N)/(1+df)) + 1`).
- **Normalization**: each vector is L2-normalized, which is what lets a plain dot product
  behave like cosine similarity and keeps the MLP's input scale stable across documents of
  very different lengths.


In [ ]:

TOKEN_RE = re.compile(r"[a-zA-Z']+")

def tokenize(text: str, use_bigrams: bool = True):
    tokens = [t.lower() for t in TOKEN_RE.findall(text)]
    if use_bigrams:
        bigrams = [f"{a}_{b}" for a, b in zip(tokens, tokens[1:])]
        return tokens + bigrams
    return tokens


class TFIDFVectorizer:
    '''Deterministic TF-IDF vectorizer implemented with base Python only
    (re, collections.Counter, math) — no scikit-learn.'''

    def __init__(self, max_features=10000, min_df=2, max_df_ratio=0.9,
                 use_bigrams=True, sublinear_tf=True):
        self.max_features = max_features
        self.min_df = min_df
        self.max_df_ratio = max_df_ratio
        self.use_bigrams = use_bigrams
        self.sublinear_tf = sublinear_tf
        self.vocab_ = {}   # token -> column index
        self.idf_ = {}     # token -> idf weight

    def fit(self, texts):
        n_docs = len(texts)
        df = Counter()
        for text in texts:
            df.update(set(tokenize(text, self.use_bigrams)))

        max_df_count = self.max_df_ratio * n_docs
        candidates = [
            (tok, c) for tok, c in df.items()
            if self.min_df <= c <= max_df_count
        ]
        # Keep the most broadly-attested terms up to the feature cap.
        candidates.sort(key=lambda x: -x[1])
        candidates = candidates[: self.max_features]

        self.vocab_ = {tok: i for i, (tok, _) in enumerate(candidates)}
        self.idf_ = {
            tok: math.log((1 + n_docs) / (1 + c)) + 1.0
            for tok, c in candidates
        }
        return self

    def _vectorize_one(self, text):
        tokens = tokenize(text, self.use_bigrams)
        counts = Counter(t for t in tokens if t in self.vocab_)
        vec = torch.zeros(len(self.vocab_))
        for tok, c in counts.items():
            tf = 1 + math.log(c) if self.sublinear_tf else float(c)
            vec[self.vocab_[tok]] = tf * self.idf_[tok]
        norm = torch.norm(vec, p=2)
        if norm > 0:
            vec = vec / norm
        return vec

    def transform(self, texts):
        return torch.stack([self._vectorize_one(t) for t in texts])

    def fit_transform(self, texts):
        self.fit(texts)
        return self.transform(texts)

    def save(self, path):
        with open(path, "w") as f:
            json.dump({
                "vocab": self.vocab_,
                "idf": self.idf_,
                "max_features": self.max_features,
                "use_bigrams": self.use_bigrams,
                "sublinear_tf": self.sublinear_tf,
            }, f)

    @classmethod
    def load(cls, path):
        with open(path) as f:
            data = json.load(f)
        vec = cls(max_features=data["max_features"],
                   use_bigrams=data["use_bigrams"],
                   sublinear_tf=data["sublinear_tf"])
        vec.vocab_ = data["vocab"]
        vec.idf_ = data["idf"]
        return vec


In [ ]:

vectorizer = TFIDFVectorizer(max_features=10000, min_df=2, max_df_ratio=0.9,
                              use_bigrams=True, sublinear_tf=True)

X_train = vectorizer.fit_transform([r["text"] for r in train_records])
y_train = torch.tensor([LABEL2IDX[r["label"]] for r in train_records], dtype=torch.long)

X_val = vectorizer.transform([r["text"] for r in val_records])
y_val = torch.tensor([LABEL2IDX[r["label"]] for r in val_records], dtype=torch.long)

X_test = vectorizer.transform([r["text"] for r in test_records])
y_test = torch.tensor([LABEL2IDX[r["label"]] for r in test_records], dtype=torch.long)

vocab_size = len(vectorizer.vocab_)
sparsity = (X_train == 0).float().mean().item()
print(f"Vocabulary size: {vocab_size}")
print(f"X_train shape:   {tuple(X_train.shape)}")
print(f"Sparsity:        {sparsity:.2%} of entries are zero")

vectorizer.save(OUT_DIR / "tfidf_vectorizer.json")



## PyTorch `Dataset` / `DataLoader`


In [ ]:

class MemoryTextDataset(Dataset):
    def __init__(self, X: torch.Tensor, y: torch.Tensor):
        self.X, self.y = X, y

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


BATCH_SIZE = 128

train_loader = DataLoader(MemoryTextDataset(X_train, y_train), batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(MemoryTextDataset(X_val, y_val), batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(MemoryTextDataset(X_test, y_test), batch_size=BATCH_SIZE, shuffle=False)



## Phase 2 — The MLP Classifier (pure PyTorch)

A small feed-forward network: `Linear -> BatchNorm -> ReLU -> Dropout`, stacked twice,
then a final `Linear` to 4 logits. `BatchNorm1d` speeds up convergence on TF-IDF's
high-dimensional, sparsity-heavy input; `Dropout` plus `weight_decay` in the optimizer
keep it from memorizing lexical quirks of any one HF source.


In [ ]:

class MemoryClassifierMLP(nn.Module):
    def __init__(self, input_dim, hidden_dims=(512, 128), num_classes=4, dropout=0.3):
        super().__init__()
        layers = []
        prev_dim = input_dim
        for h in hidden_dims:
            layers += [
                nn.Linear(prev_dim, h),
                nn.BatchNorm1d(h),
                nn.ReLU(),
                nn.Dropout(dropout),
            ]
            prev_dim = h
        layers.append(nn.Linear(prev_dim, num_classes))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)


model = MemoryClassifierMLP(input_dim=vocab_size, hidden_dims=(512, 128),
                             num_classes=len(LABELS), dropout=0.3).to(DEVICE)
print(model)
n_params = sum(p.numel() for p in model.parameters())
print(f"\nTotal trainable parameters: {n_params:,}")



## Training loop — early stopping + LR scheduling

Standard supervised training: `CrossEntropyLoss` + `AdamW`, with `ReduceLROnPlateau`
halving the learning rate when validation loss stalls, and early stopping that restores
the best-validation-loss weights once patience is exhausted.


In [ ]:

def evaluate_loss_acc(model, loader):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for xb, yb in loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            logits = model(xb)
            loss = criterion(logits, yb)
            total_loss += loss.item() * xb.size(0)
            correct += (logits.argmax(1) == yb).sum().item()
            total += xb.size(0)
    return total_loss / total, correct / total


criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=2)

EPOCHS = 40
PATIENCE = 6

best_val_loss = float("inf")
best_state = None
patience_ctr = 0
history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}

for epoch in range(1, EPOCHS + 1):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for xb, yb in train_loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * xb.size(0)
        correct += (logits.argmax(1) == yb).sum().item()
        total += xb.size(0)

    train_loss, train_acc = running_loss / total, correct / total
    val_loss, val_acc = evaluate_loss_acc(model, val_loader)
    scheduler.step(val_loss)

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["train_acc"].append(train_acc)
    history["val_acc"].append(val_acc)

    print(f"Epoch {epoch:02d} | train_loss {train_loss:.4f} acc {train_acc:.4f} "
          f"| val_loss {val_loss:.4f} acc {val_acc:.4f}")

    if val_loss < best_val_loss - 1e-4:
        best_val_loss = val_loss
        best_state = {k: v.clone() for k, v in model.state_dict().items()}
        patience_ctr = 0
    else:
        patience_ctr += 1
        if patience_ctr >= PATIENCE:
            print(f"\nEarly stopping at epoch {epoch} (best val_loss={best_val_loss:.4f})")
            break

model.load_state_dict(best_state)


In [ ]:

# --- Training curves ---
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].plot(history["train_loss"], label="train")
axes[0].plot(history["val_loss"], label="val")
axes[0].set_title("Loss"); axes[0].set_xlabel("epoch"); axes[0].legend()

axes[1].plot(history["train_acc"], label="train")
axes[1].plot(history["val_acc"], label="val")
axes[1].set_title("Accuracy"); axes[1].set_xlabel("epoch"); axes[1].legend()

plt.tight_layout()
plt.show()



## Evaluation — manual metrics (no sklearn)

Confusion matrix, per-class precision/recall/F1, and overall accuracy, all computed with
plain `torch` tensor operations.


In [ ]:

def confusion_matrix(y_true, y_pred, num_classes):
    cm = torch.zeros(num_classes, num_classes, dtype=torch.long)
    for t, p in zip(y_true, y_pred):
        cm[t, p] += 1
    return cm


def classification_report(y_true, y_pred, labels):
    num_classes = len(labels)
    cm = confusion_matrix(y_true, y_pred, num_classes)
    report = {}
    for i, label in enumerate(labels):
        tp = cm[i, i].item()
        fp = cm[:, i].sum().item() - tp
        fn = cm[i, :].sum().item() - tp
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
        support = cm[i, :].sum().item()
        report[label] = {"precision": precision, "recall": recall, "f1": f1, "support": support}
    accuracy = torch.diag(cm).sum().item() / cm.sum().item()
    return report, accuracy, cm


model.eval()
with torch.no_grad():
    test_logits = model(X_test.to(DEVICE))
    test_preds = test_logits.argmax(1).cpu()

report, accuracy, cm = classification_report(y_test.tolist(), test_preds.tolist(), LABELS)

print(f"Test accuracy: {accuracy:.4f}\n")
print(f"{'class':10s} {'precision':>10s} {'recall':>10s} {'f1':>10s} {'support':>10s}")
for label, m in report.items():
    print(f"{label:10s} {m['precision']:10.3f} {m['recall']:10.3f} {m['f1']:10.3f} {m['support']:10d}")


In [ ]:

# --- Confusion matrix heatmap ---
fig, ax = plt.subplots(figsize=(5, 4.5))
im = ax.imshow(cm.numpy(), cmap="Blues")
ax.set_xticks(range(len(LABELS))); ax.set_xticklabels(LABELS, rotation=45, ha="right")
ax.set_yticks(range(len(LABELS))); ax.set_yticklabels(LABELS)
ax.set_xlabel("Predicted"); ax.set_ylabel("True")
ax.set_title(f"Confusion matrix (test acc={accuracy:.3f})")

for i in range(len(LABELS)):
    for j in range(len(LABELS)):
        val = cm[i, j].item()
        color = "white" if val > cm.max().item() / 2 else "black"
        ax.text(j, i, str(val), ha="center", va="center", color=color)

fig.colorbar(im, ax=ax, shrink=0.8)
plt.tight_layout()
plt.show()



## Save artifacts

The trained model weights, the TF-IDF vocabulary/IDF table, and the label map are saved
so the classifier can be loaded standalone by the downstream Phase 3 (SPO extraction /
storage routing) and Phase 4 (conflict resolution) components without retraining.


In [ ]:

torch.save(model.state_dict(), OUT_DIR / "memory_classifier_mlp.pt")
vectorizer.save(OUT_DIR / "tfidf_vectorizer.json")
with open(OUT_DIR / "label_map.json", "w") as f:
    json.dump({"labels": LABELS, "label2idx": LABEL2IDX}, f, indent=2)

# Also persist the architecture hyperparams needed to reconstruct the model at load time.
with open(OUT_DIR / "model_config.json", "w") as f:
    json.dump({
        "input_dim": vocab_size,
        "hidden_dims": [512, 128],
        "num_classes": len(LABELS),
        "dropout": 0.3,
    }, f, indent=2)

print("Saved to:", [p.name for p in OUT_DIR.iterdir()])



## Inference demo — classify & route

A single deterministic function that takes raw text, vectorizes it with the fitted
TF-IDF vectorizer, runs it through the trained MLP, and returns the predicted memory
type plus a softmax confidence score. This confidence score is exactly the signal Phase 4
(adversarial conflict resolution) will use downstream to decide whether a new fact should
overwrite, supersede, or be rejected against an existing memory entry.


In [ ]:

@torch.no_grad()
def classify_and_route(text: str, model=model, vectorizer=vectorizer):
    model.eval()
    vec = vectorizer.transform([text]).to(DEVICE)
    logits = model(vec)
    probs = torch.softmax(logits, dim=1).squeeze(0)
    pred_idx = int(probs.argmax())
    return {
        "text": text,
        "predicted_memory": IDX2LABEL[pred_idx],
        "confidence": round(probs[pred_idx].item(), 4),
        "distribution": {IDX2LABEL[i]: round(p.item(), 4) for i, p in enumerate(probs)},
    }


demo_examples = [
    "I am an introvert who prefers quiet evenings at home.",                 # personal
    "The mitochondria is the powerhouse of the cell.",                       # semantic
    "Remind me to call the dentist at 4pm today.",                          # working
    "Last summer I backpacked through Portugal with my cousin.",            # episodic
]

for ex in demo_examples:
    result = classify_and_route(ex)
    print(f"[{result['predicted_memory']:10s} | conf={result['confidence']:.3f}] {ex}")



## Next steps (Phase 3 / Phase 4 — not implemented in this notebook)

This notebook covers Phase 1 (aggregation) and Phase 2 (classification). The saved
artifacts in `artifacts/` (`memory_classifier_mlp.pt`, `tfidf_vectorizer.json`,
`label_map.json`, `model_config.json`) are the handoff point for:

- **Phase 3 — Flattened SPO extraction**: run each classified string through a rule-based
  spaCy dependency parse, route the extracted Subject-Predicate-Object triple into the
  schema that matches its predicted `label` (`personal` / `semantic` / `episodic` / `working`).
- **Phase 4 — Adversarial conflict resolution**: use `confidence` (from `classify_and_route`)
  alongside `timestamp` and memory type to decide, deterministically, whether an incoming
  fact supersedes an existing one or gets flagged as a low-confidence distractor.
